# 🚀 InstantMesh Colab GPU Inference Server
### Sketch2Shape AI - High-Performance 3D Reconstruction

This Google Colab notebook provides cloud GPU acceleration for **InstantMesh** on free NVIDIA T4 (15GB VRAM) or A100 GPUs, allowing full-resolution 3D mesh reconstruction from single sketches or images without local VRAM bottlenecks.

#### Features:
- Automated PyTorch 2.1 + CUDA 12.1 setup
- Automated download of official InstantMesh checkpoints (`TencentARC/InstantMesh`)
- FastAPI inference server with `/reconstruct` endpoint
- Public tunnel generation (Cloudflared / Ngrok) for direct connection to local Sketch2Shape UI

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

In [ ]:
# Step 2: Clone InstantMesh Repository
!git clone https://github.com/TencentARC/InstantMesh.git
%cd InstantMesh

In [ ]:
# Step 3: Install Required Dependencies
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn pyngrok trimesh rembg python-multipart

In [ ]:
# Step 4: Download Pretrained Checkpoints from Hugging Face
import os
from huggingface_hub import hf_hub_download

os.makedirs('ckpts', exist_ok=True)
print('Downloading diffusion checkpoint...')
hf_hub_download(repo_id='TencentARC/InstantMesh', filename='diffusion_pytorch_model.bin', local_dir='ckpts')
print('Downloading InstantMesh-Large checkpoint...')
hf_hub_download(repo_id='TencentARC/InstantMesh', filename='instant_mesh_large.ckpt', local_dir='ckpts')
print('Checkpoints ready!')

In [ ]:
# Step 5: Start FastAPI Server with Ngrok/Cloudflared Tunnel
import io
import torch
import trimesh
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import Response

app = FastAPI(title='InstantMesh API')

@app.get('/health')
def health():
    return {'status': 'healthy', 'gpu': torch.cuda.get_device_name(0), 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)}

@app.post('/reconstruct')
async def reconstruct(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert('RGB')
    image.save('input.png')
    
    # Run official inference script
    !python run.py configs/instant-mesh-large.yaml input.png --output_path outputs/
    
    # Load generated mesh and convert to GLB
    mesh_path = 'outputs/instant-mesh-large/meshes/input.obj'
    mesh = trimesh.load(mesh_path)
    glb_bytes = mesh.export(file_type='glb')
    return Response(content=glb_bytes, media_type='model/gltf-binary')

print('Server ready to mount via uvicorn!')